In [1]:
import numpy as np
import pandas as pd

In [2]:
data_path = "mock_output.csv"

In [3]:
data = pd.read_csv(data_path)

In [4]:
data = data.rename(columns={"used_for_parameter_estimation" : "parameter"})
data = data.rename(columns={"bob_measured_quadrature" : "bob_basis"})

In [6]:
# in the real data we won't already have this so simulate we don't
data = data.drop(["alice_value_for_bob_basis"], axis=1)

### Keep only Alice values corresponding to Bob's measurements

In [7]:
# keep track of when bob measured X and when he measured P
bob_x_idxs = data["bob_basis"] == "X"
bob_p_idxs = data["bob_basis"] == "P"

In [8]:
alice_value_x = data[bob_x_idxs]["alice_x"]
alice_value_p = data[bob_p_idxs]["alice_p"]

In [9]:
alice_value = pd.concat([alice_value_x, alice_value_p])

In [10]:
data.insert(6, "alice_value", alice_value)

In [11]:
data.head()

,pulse_id,alice_x,alice_p,bob_basis,bob_value,parameter,alice_value
0,0,0.609434,-0.903902,X,-2.315105,False,0.609434
1,1,-2.079968,-1.331755,X,-2.134312,False,-2.079968
2,2,1.500902,0.868020,X,0.574552,False,1.500902
3,3,1.881129,0.503709,X,1.668453,False,1.881129
4,4,-3.902070,-2.809583,X,-2.570418,False,-3.902070


### Estimate information rate between Alice and Bob

In [12]:
# only use subset designated for parameter estimation
parameter_subset = data[data["parameter"] == True]

A = parameter_subset["alice_value"]
B = parameter_subset["bob_value"]

### Channel gain

### $g = \frac{\text{Cov}(A,B)}{\text{Var}(A)}$

In [33]:
channel_gain = np.cov(A, B)[0,1] / np.var(A)
print(f"Channel gain g : {channel_gain}")

Channel gain g : 0.8165714135341845


### Noise variance

Variance of the noise that remains after removing the part of Bob's measurement that is explained by Alice's value

#### $\sigma_n^2 = \text{Var}(B - gA)$

In [34]:
noise_var = np.var(B - channel_gain * A)
print(f"The noise variance is : {noise_var}")

The noise variance is : 1.056759005609136


### Signal to noise ratio

### $\Sigma_B = \frac{g^2 \text{Var}(A)}{\sigma_n^2} $

In [35]:
bob_snr = (channel_gain * np.var(A)) / noise_var
print(f"Bob's signal to noise ratio : {bob_snr}")

Bob's signal to noise ratio : 2.9271795225417563


### Information rate between Alice and Bob

### $I_{AB} = \frac{1}{2} \log_2 (1 + \Sigma_B)$

In [37]:
I_AB = 0.5 * (np.log2(1 + bob_snr))
print(f"I_AB = {I_AB}")

I_AB = 0.9867467746019641


### Upper bound on Eve's information